# Evidence Grounding in Qwen2.5-VL: Initial Experiments

Does the model's answer depend on the region of the image that actually contains the evidence?

For 500 GQA image–question pairs, each with a single annotated evidence bounding box, we score the
ground-truth answer under several views of the image:

| Condition | Image given to the model | Question it answers |
|---|---|---|
| Question only | none | How much can be answered from language priors? |
| Full image | original | Baseline |
| Evidence only | everything outside the box grayed out | **Sufficiency**: is the box enough? |
| Evidence removed | the box grayed out | **Necessity**: is the box needed? |
| Irrelevant region removed | a same-size, non-overlapping box grayed out | Control for necessity |

**Metric.** We don't generate answers. We teacher-force the ground-truth answer and take the mean
per-token log-probability the model assigns to it. Results are reported as NLL (the negated value),
so lower is better.

## Setup

In [ ]:
# Mount Google Drive (the data and results live there) and move into the project folder.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/CS8803-VLM

In [ ]:
import json
import os
import random
import urllib.request
from urllib.error import HTTPError

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image, UnidentifiedImageError
from scipy.stats import wilcoxon
from tqdm.auto import tqdm
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

# Paths (relative to the project folder above) and the model checkpoint.
DATA_PATH = "data/gqa_cot_train.jsonl"
IMAGE_DIR = "data/image_subset"
RESULTS_DIR = "results"
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

## Data: sample 500 GQA examples

For these initial experiments we keep only examples that have:
- exactly one evidence box, so each masking condition is well defined,
- a box covering 5–50% of the image, so masking it is neither trivial nor destructive,
- an answer of at most two words, which suits the keyword-only prompt.

In [ ]:
def area_frac(r):
    """Fraction of the image covered by the example's (first) evidence box."""
    x1, y1, x2, y2 = r["bboxs"][0]
    return (x2 - x1) * (y2 - y1) / (r["width"] * r["height"])


with open(DATA_PATH) as f:
    rows = [json.loads(line) for line in f]

good = [
    r for r in rows
    if len(r["bboxs"]) == 1
    and 0.05 <= area_frac(r) <= 0.5
    and len(r["answer"].split()) <= 2
]

# Seed the global RNG. The irrelevant-region control later draws from this same RNG,
# so its random boxes are reproducible too (as long as cells run in order).
random.seed(0)
subset = random.sample(good, 500)

In [ ]:
# Download only the 500 sampled images instead of the full ~20 GB GQA image set.
# Stanford splits the Visual Genome images across two folders, so an image that 404s
# in the first folder is retried in the second.
VG_URL_1 = "https://cs.stanford.edu/people/rak248/VG_100K"
VG_URL_2 = "https://cs.stanford.edu/people/rak248/VG_100K_2"


def download_image(filename, target_path):
    """Download one image, falling back to the second folder on a 404. Returns True on success."""
    try:
        urllib.request.urlretrieve(f"{VG_URL_1}/{filename}", target_path)
        return True
    except HTTPError as e:
        if e.code != 404:
            print(f"\n[Error] HTTP Error {e.code} for {filename}: {e.reason}")
            return False

    # Not in folder 1, so try folder 2.
    try:
        urllib.request.urlretrieve(f"{VG_URL_2}/{filename}", target_path)
        return True
    except Exception as fallback_err:
        print(f"\n[404 Both] Failed to download {filename} from either folder: {fallback_err}")
        return False


os.makedirs(IMAGE_DIR, exist_ok=True)
failed_images = []
success_count = 0
for example in tqdm(subset, desc="Downloading GQA Subsample Images"):
    filename = example["image"]
    target_path = os.path.join(IMAGE_DIR, filename)

    # Already downloaded on a previous run: count it and move on.
    if os.path.exists(target_path) or download_image(filename, target_path):
        success_count += 1
    else:
        failed_images.append(filename)

print(f"\nFinished! Successfully verified/downloaded {success_count}/{len(subset)} images.")

## Model and scoring helpers

In [ ]:
# Load Qwen2.5-VL-7B in bfloat16; device_map="auto" places it on the available GPU.
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
)
model.eval()
processor = AutoProcessor.from_pretrained(MODEL_ID)

In [ ]:
# Few-shot instruction that pushes the model to answer with a bare keyword, so that
# its answers match the short GQA ground-truth answers we score against.
INSTRUCTION = (
    "Answer the question with only the key word or short phrase. "
    "Do not write a full sentence and do not use punctuation.\n\n"
    "Q: What is the man holding?\nA: umbrella\n\n"
    "Q: What color is the bus?\nA: red\n\n"
)


def make_inputs(processor, question, image=None):
    """Build a chat-formatted prompt that ends exactly where the assistant's answer begins.

    With image=None the prompt is text only (used for the question-only condition).
    """
    content = [{"type": "text", "text": INSTRUCTION + f"Q: {question}\nA:"}]
    if image is not None:
        content = [{"type": "image"}] + content
    messages = [{"role": "user", "content": content}]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    if image is not None:
        return processor(text=[prompt], images=[image], return_tensors="pt")
    return processor(text=[prompt], return_tensors="pt")


@torch.no_grad()
def generate_answer(model, processor, question, image=None, max_new_tokens=8):
    """Greedy-decode a short free-form answer. Handy for spot checks; the experiments use answer_logprob."""
    inputs = make_inputs(processor, question, image).to(model.device)
    out = model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False,
        temperature=None, top_p=None, top_k=None,
    )
    new_tokens = out[0, inputs["input_ids"].shape[1]:]  # drop the echoed prompt
    return processor.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [ ]:
@torch.no_grad()  # inference only; skipping autograd saves a lot of GPU memory
def answer_logprob(model, processor, image, question, answer):
    """Score how likely the model finds `answer`, using teacher forcing.

    Returns (sum, mean) of the log-probabilities of the answer tokens. The experiments use the
    mean (index [1]) so that answers of different token lengths are comparable.
    """
    inputs = make_inputs(processor, question, image)
    n_prompt = inputs["input_ids"].shape[1]

    # Append the tokenized ground-truth answer right after the prompt.
    ans_ids = processor.tokenizer(answer, add_special_tokens=False, return_tensors="pt")["input_ids"]
    inputs["input_ids"] = torch.cat([inputs["input_ids"], ans_ids], dim=1)
    inputs = inputs.to(model.device)

    out = model(
        input_ids=inputs["input_ids"],
        pixel_values=inputs.get("pixel_values"),
        image_grid_thw=inputs.get("image_grid_thw"),
    )
    logits = out.logits[0]
    ids = inputs["input_ids"][0]

    # Row t of the logits predicts token t+1, so pair logits[:-1] with ids[1:] and look up the
    # log-prob of each actual next token.
    logp = torch.log_softmax(logits[:-1].float(), -1).gather(1, ids[1:, None])[:, 0]

    # Keep only the positions that predict answer tokens (the first is predicted by the last prompt token).
    ans_logp = logp[n_prompt - 1:]
    return ans_logp.sum().item(), ans_logp.mean().item()

In [ ]:
FILL = (124, 116, 104)  # the same neutral gray is used in every masked view


def mask_outside(img, box):
    """Evidence only: gray out everything except `box`."""
    out = Image.new("RGB", img.size, FILL)  # gray canvas the size of the original
    out.paste(img.crop(box), box[:2])       # paste the box back at its original position
    return out


def mask_evidence(img, box):
    """Evidence removed: gray out only `box`."""
    out = img.copy()
    out.paste(FILL, box)
    return out


def random_box_outside(img_size, box):
    """Pick a random box the same size as `box` that fits in the image and doesn't overlap `box`.

    Valid top-left corners fall into four regions: fully left of the box, fully right of it,
    or above or below it (limited to the columns that overlap the box horizontally, so they don't
    double count the left and right regions). A region is chosen with probability proportional
    to its area, so every valid placement is equally likely. Each region is (x_min, x_max, y_min, y_max).
    Raises RuntimeError if no valid placement exists.
    """
    W, H = img_size
    l, u, r, b = box
    w, h = r - l, b - u

    mx0, mx1 = max(0, l - w + 1), min(W - w, r - 1)  # x positions that overlap the box horizontally

    regions = [
        (0, l - w, 0, H - h),   # left, full height
        (r, W - w, 0, H - h),   # right, full height
        (mx0, mx1, 0, u - h),   # above, middle columns only
        (mx0, mx1, b, H - h),   # below, middle columns only
    ]
    regions = [g for g in regions if g[0] <= g[1] and g[2] <= g[3]]  # drop empty regions
    if not regions:
        raise RuntimeError("No non-overlapping placement exists.")

    areas = [(g[1] - g[0] + 1) * (g[3] - g[2] + 1) for g in regions]
    x0, x1, y0, y1 = random.choices(regions, weights=areas, k=1)[0]
    x, y = random.randint(x0, x1), random.randint(y0, y1)
    return (x, y, x + w, y + h)


def mask_irrelevant_region(img, box):
    """Control: gray out a random same-size region away from the evidence.

    Returns (masked_image, random_box), or (None, None) if no such region fits.
    """
    try:
        rand_box = random_box_outside(img.size, box)
    except RuntimeError:
        return None, None
    return mask_evidence(img, rand_box), rand_box

In [ ]:
def load_image(example):
    """Open an example's downloaded image as RGB."""
    return Image.open(os.path.join(IMAGE_DIR, example["image"])).convert("RGB")


def ground_truth(example):
    """Normalized ground-truth answer string that gets scored."""
    return example["answer"].strip().lower()


def run_masked_condition(examples, mask_fn, name):
    """Score every example under one masked view of its image.

    mask_fn(img, box) returns the masked image, or None if the mask can't be applied to that image.
    Results go in columns f"{name}_logprob" (mean per token) and f"{name}_logprob_sum".

    Returns (results_df, errored_images, unmaskable_images): errored images couldn't be opened,
    and unmaskable images are ones where mask_fn returned None.
    """
    rows, errored, unmaskable = [], [], []
    for example in tqdm(examples):
        try:
            img = load_image(example)
        except (UnidentifiedImageError, OSError):
            errored.append(example["image"])
            continue

        masked = mask_fn(img, tuple(example["bboxs"][0]))
        if masked is None:
            unmaskable.append(example["image"])
            continue

        gt = ground_truth(example)
        total_lp, mean_lp = answer_logprob(model, processor, masked, example["question"], gt)
        rows.append({
            "image": example["image"],
            "question": example["question"],
            "answer": gt,
            f"{name}_logprob": mean_lp,
            f"{name}_logprob_sum": total_lp,
        })
        print(f"{example['question'][:45]:45s} gt={gt:12s} {name}={mean_lp:6.2f}")

    return pd.DataFrame(rows), errored, unmaskable

In [ ]:
# Plotting helpers shared by the analysis cells below.

def scatter_vs_diagonal(ax, x, y, xlabel, ylabel, title):
    """Per-example scatter with a y = x reference line. Points above the line have a larger y."""
    ax.scatter(x, y, alpha=0.4, s=15)
    lo, hi = min(x.min(), y.min()), max(x.max(), y.max())
    ax.plot([lo, hi], [lo, hi], "r--")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)


def hist_with_zero_line(ax, values, xlabel, title):
    """Histogram of per-example differences with a dashed line at 0."""
    ax.hist(values, bins=30, edgecolor="black")
    ax.axvline(0, color="red", linestyle="--")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Number of Examples")
    ax.set_title(title)


def plot_condition_overview(logprobs, label, titles):
    """Three panels for a single condition: the log-prob histogram, the NLL histogram, and NLL by example."""
    nll = -logprobs
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].hist(logprobs, bins=30, edgecolor="black")
    axes[0].set_xlabel(f"{label} Log Probability")
    axes[0].set_ylabel("Number of Examples")
    axes[0].set_title(titles[0])

    axes[1].hist(nll, bins=30, edgecolor="black")
    axes[1].set_xlabel(f"{label} NLL")
    axes[1].set_ylabel("Number of Examples")
    axes[1].set_title(titles[1])

    axes[2].scatter(range(len(nll)), nll, alpha=0.4, s=15)
    axes[2].axhline(nll.mean(), color="red", linestyle="--", label=f"Mean NLL = {nll.mean():.3f}")
    axes[2].set_xlabel("Example Index")
    axes[2].set_ylabel(f"{label} NLL")
    axes[2].set_title(titles[2])
    axes[2].legend()

    plt.tight_layout()
    plt.show()

## Experiment 1: Question only, full image, and evidence only

Comparing question only with the full image shows how much the model relies on vision at all.
Comparing evidence only with the full image tests **sufficiency**: if the box holds the evidence,
hiding everything else shouldn't hurt.

In [ ]:
full_results, qonly_results, evidence_only_results = [], [], []
for example in subset:
    # Note: this loop only skips UnidentifiedImageError. Later loops also skip other OSErrors.
    try:
        img = load_image(example)
    except UnidentifiedImageError:
        continue

    question, gt = example["question"], ground_truth(example)
    box = tuple(example["bboxs"][0])  # the filter guarantees exactly one box per example

    # [1] picks the mean per-token log-prob out of answer_logprob's (sum, mean).
    _, full = answer_logprob(model, processor, img, question, gt)
    _, qonly = answer_logprob(model, processor, None, question, gt)
    _, box_only = answer_logprob(model, processor, mask_outside(img, box), question, gt)

    full_results.append(full)
    qonly_results.append(qonly)
    evidence_only_results.append(box_only)
    print(f"{question[:45]:45s} gt={gt:12s} full={full:6.2f} qonly={qonly:6.2f} evidence_only={box_only:6.2f}")

print("Mean NLL for Full Image Across 500 Examples: ", -np.mean(full_results))
print("Mean NLL for Question Only Across 500 Examples: ", -np.mean(qonly_results))
print("Mean NLL for Evidence Only Across 500 Examples: ", -np.mean(evidence_only_results))

In [ ]:
full = np.array(full_results)
qonly = np.array(qonly_results)
evidence = np.array(evidence_only_results)
difference = evidence - full  # > 0: isolating the evidence made the answer *more* likely

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
scatter_vs_diagonal(axes[0], full, evidence,
                    "Full Image Log Probability", "Evidence Only Log Probability",
                    "Does Evidence Alone Preserve Answers?")
scatter_vs_diagonal(axes[1], qonly, full,
                    "Question Only Log Probability", "Full Image Log Probability",
                    "How Much Does Vision Help?")
hist_with_zero_line(axes[2], difference,
                    "Evidence Only - Full Image Log Probability", "Effect of Isolating Evidence")
plt.tight_layout()
plt.show()

print(f"Examples: {len(full)}")
print(f"Mean full-image NLL: {-full.mean():.3f}")
print(f"Mean evidence-only NLL: {-evidence.mean():.3f}")
print(f"Mean question-only NLL: {-qonly.mean():.3f}")
print(f"Evidence-only improves likelihood: {(difference > 0).mean():.1%}")

## Experiment 2: Evidence removed (necessity)

Gray out the evidence box itself. If the model really uses that region, the ground-truth answer
should become much less likely.

In [ ]:
# mask_evidence always succeeds, so the third return value (unmaskable images) is always empty.
df, skipped, _ = run_masked_condition(subset, mask_evidence, "evidence_removed")

print("\nSuccessfully evaluated:", len(df))
print("Skipped images:", skipped)
print("Mean evidence-removed NLL:", -df["evidence_removed_logprob"].mean())

os.makedirs(RESULTS_DIR, exist_ok=True)
df.to_csv(f"{RESULTS_DIR}/evidence_removed_results.csv", index=False)

In [ ]:
removed = df["evidence_removed_logprob"].to_numpy()
plot_condition_overview(removed, "Evidence Removed", (
    "How Confident Is the Model Without Evidence?",
    "Ground-Truth Answer NLL Without Evidence",
    "Evidence-Removed Performance Across Examples",
))

print(f"Examples: {len(removed)}")
print(f"Mean evidence-removed NLL: {-removed.mean():.3f}")

In [ ]:
# Join Experiment 1's per-example results with Experiment 2's so all four conditions line up.
# Experiment 1's lists don't record which images were skipped, so rebuild that example list by
# dropping the images Experiment 2 skipped. This assumes both loops skipped the same images.
valid_examples = [ex for ex in subset if ex["image"] not in skipped]

sufficiency_df = pd.DataFrame({
    "image": [ex["image"] for ex in valid_examples],
    "question": [ex["question"] for ex in valid_examples],
    "answer": [ground_truth(ex) for ex in valid_examples],
    "full_logprob": full_results,
    "evidence_only_logprob": evidence_only_results,
    "qonly_logprob": qonly_results,
})

comparison = sufficiency_df.merge(
    df, on=["image", "question", "answer"], how="inner", validate="one_to_one"
)

full = comparison["full_logprob"].to_numpy()
removed = comparison["evidence_removed_logprob"].to_numpy()
difference = full - removed  # > 0: removing the evidence hurt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
scatter_vs_diagonal(axes[0], full, removed,
                    "Full Image Log Probability", "Evidence Removed Log Probability",
                    "Does Removing Evidence Hurt?")
hist_with_zero_line(axes[1], difference,
                    "Full - Evidence Removed Log Probability", "Necessity Effect Distribution")

# Mean NLL for each condition, roughly from worst to best.
conditions = {
    "Question Only": "qonly_logprob",
    "Evidence Removed": "evidence_removed_logprob",
    "Evidence Only": "evidence_only_logprob",
    "Full Image": "full_logprob",
}
axes[2].bar(list(conditions), [-comparison[col].mean() for col in conditions.values()])
axes[2].set_ylabel("Mean NLL (Lower Is Better)")
axes[2].set_title("Overall Performance Comparison")
axes[2].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

print(f"Examples compared: {len(comparison)}")
print(f"Mean full-image NLL: {-full.mean():.3f}")
print(f"Mean evidence-removed NLL: {-removed.mean():.3f}")
print(f"Mean necessity effect: {difference.mean():.3f}")
print(f"Evidence removal hurts in {(difference > 0).mean():.1%} of examples")

## Experiment 3: Irrelevant region removed (control)

Graying out *any* region of the same size might hurt simply because pixels are missing. To control
for that, mask a random box of the same size that doesn't overlap the evidence. Examples where no
such box fits are skipped.

In [ ]:
# mask_irrelevant_region returns (image, box); we only need the image (None if no box fits).
irrelevant_region_removed_df, errored_images, skipped_irrelevant_region = run_masked_condition(
    subset, lambda img, box: mask_irrelevant_region(img, box)[0], "irrelevant_removed"
)

print("\nSuccessfully evaluated:", len(irrelevant_region_removed_df))
print("No found non-overlapping region images:", skipped_irrelevant_region)
print("Other errored images", errored_images)
print("Mean irrelevant-region-removed NLL:",
      -irrelevant_region_removed_df["irrelevant_removed_logprob"].mean())

irrelevant_region_removed_df.to_csv(f"{RESULTS_DIR}/irrelevant_region_removed_results.csv", index=False)

In [ ]:
irrelevant_region_removed = irrelevant_region_removed_df["irrelevant_removed_logprob"].to_numpy()
plot_condition_overview(irrelevant_region_removed, "Irrelevant Region Removed", (
    "How Confident Is the Model With an Irrelevant Region Removed?",
    "Ground-Truth Answer NLL With An Irrelevant Region Removed",
    "Irrelevant Region-Removed Performance Across Examples",
))

print(f"Examples: {len(irrelevant_region_removed)}")
print(f"Mean Irrelevant Region-removed NLL: {-irrelevant_region_removed.mean():.3f}")

## Control-adjusted necessity

Pair each example's evidence-removed NLL with its irrelevant-region-removed NLL. A positive
difference means hiding the evidence hurts more than hiding a random region of the same size.
We report a bootstrap CI for the mean difference and a Wilcoxon signed-rank test.

In [ ]:
# --- config: adjust to your variable/column names ---
EVI_DF = pd.read_csv(f"{RESULTS_DIR}/evidence_removed_results.csv")
EVI_COL = "evidence_removed_logprob"
IRR_DF = irrelevant_region_removed_df
IRR_COL = "irrelevant_removed_logprob"
keys = ["image", "question"]

# Area fraction of each evidence box, used to check whether the effect depends on mask size.
# Unlike area_frac() above, this clips the box to the image bounds first.
area_rows = []
for ex in subset:
    W, H = ex["width"], ex["height"]
    x1, y1, x2, y2 = ex["bboxs"][0]
    x1, y1, x2, y2 = max(0, x1), max(0, y1), min(W, x2), min(H, y2)
    area_rows.append({
        "image": ex["image"],
        "question": ex["question"],
        "area_frac": (x2 - x1) * (y2 - y1) / (W * H),
    })
area_df = pd.DataFrame(area_rows).drop_duplicates(keys)

# The inner join keeps only examples that were scored in BOTH conditions.
merged = (
    IRR_DF[keys + [IRR_COL]]
    .merge(EVI_DF[keys + [EVI_COL]], on=keys, how="inner")
    .merge(area_df, on=keys, how="left")
)

irr_nll = -merged[IRR_COL].to_numpy()
evi_nll = -merged[EVI_COL].to_numpy()
delta = evi_nll - irr_nll  # > 0: hiding the evidence hurts more than hiding a random region

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1: NLL distributions for the two conditions, on shared bins
bins = np.linspace(min(irr_nll.min(), evi_nll.min()), max(irr_nll.max(), evi_nll.max()), 30)
axes[0].hist(irr_nll, bins=bins, alpha=0.6, label=f"Irrelevant removed (mean {irr_nll.mean():.3f})")
axes[0].hist(evi_nll, bins=bins, alpha=0.6, label=f"Evidence removed (mean {evi_nll.mean():.3f})")
axes[0].set_xlabel("NLL of ground-truth answer")
axes[0].set_ylabel("Number of examples")
axes[0].set_title("NLL by masking condition")
axes[0].legend()

# 2: paired per-example difference
axes[1].hist(delta, bins=30, edgecolor="black")
axes[1].axvline(0, color="black", linewidth=1)
axes[1].axvline(delta.mean(), color="red", linestyle="--", label=f"Mean Δ = {delta.mean():.3f}")
axes[1].set_xlabel("Evidence NLL − Irrelevant NLL")
axes[1].set_ylabel("Number of examples")
axes[1].set_title("Control-Adjusted Necessity Distribution (positive = evidence matters)")
axes[1].legend()

# 3: does the effect depend on how much of the image was masked?
axes[2].scatter(merged["area_frac"], delta, alpha=0.4, s=15)
axes[2].axhline(0, color="black", linewidth=1)
axes[2].set_xlabel("Masked area fraction")
axes[2].set_ylabel("Evidence NLL − Irrelevant NLL")
axes[2].set_title("Effect vs. size of masked region")

plt.tight_layout()
plt.show()

# --- summary stats ---
# 95% CI for the mean difference from 5000 bootstrap resamples, plus a non-parametric paired test.
rng_boot = np.random.default_rng(0)
boot = [rng_boot.choice(delta, len(delta)).mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
stat, p = wilcoxon(delta)

print(f"Paired examples:               {len(merged)}")
print(f"Mean irrelevant-removed NLL:   {irr_nll.mean():.3f}")
print(f"Mean evidence-removed NLL:     {evi_nll.mean():.3f}")
print(f"Control-Adjusted Necessity (evidence − irrelevant): {delta.mean():.3f}  (95% bootstrap CI [{lo:.3f}, {hi:.3f}])")
print(f"Median Necessity:                      {np.median(delta):.3f}")
print(f"Fraction where evidence hurts more: {(delta > 0).mean():.2%}")
print(f"Wilcoxon signed-rank p-value:  {p:.3g}")